# SAGE — synthetic trajectory generation on Colab

Runs a 30B-class uncensored GGUF model via `llama-cpp-python` to produce conversation-level safety training data. Writes a pending JSONL per bucket to Drive. You human-review locally afterward and merge with `synthetic.cli merge`.

**Two models selectable (cell 4):**
- `qwen36` — `HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive` — 35B MoE, 3B active per token, 0/465 refusals. Fastest.
- `gemma4` — `llmfan46/gemma-4-31B-it-uncensored-heretic-GGUF` — 31B dense, 10/100 refusals, stronger creative/dialogue quality.

**Auto-selected quant based on detected VRAM:**

| VRAM | Quant | Quality | Weights |
|---|---|---|---|
| ≥ 70 GB (A100 80GB) | **Q8_0** | near-lossless | ~33–36 GB |
| ≥ 35 GB (A100 40GB) | Q4_K_M | standard | ~18–20 GB |
| ≥ 22 GB (L4 24GB) | Q4_K_M | standard, smaller ctx | ~18–20 GB |

Override manually in cell 4 if you want to force a specific quant (e.g. Q6_K on 40GB).

**Incremental saving is baked in.** Each batch of generated examples is appended to the pending JSONL immediately. If Colab disconnects mid-run, re-run cell 9 — it resumes from where the file left off, no work is lost.

**Before running:**
1. **Runtime → Change runtime type → GPU → A100**
2. Mount Drive in cell 1 so outputs survive disconnects
3. First session downloads model (~20–36 GB depending on quant) to Drive cache — ~20–30 min

## 1. Mount Drive

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
WORK_DIR = '/content/drive/MyDrive/sage'
PENDING_DIR = f'{WORK_DIR}/synthetic/pending'
MODEL_CACHE = f'{WORK_DIR}/models'
os.makedirs(PENDING_DIR, exist_ok=True)
os.makedirs(MODEL_CACHE, exist_ok=True)
print(f'work dir:     {WORK_DIR}')
print(f'pending dir:  {PENDING_DIR}')
print(f'model cache:  {MODEL_CACHE}')

## 2. GPU detection + auto-tune

Picks batch / context / parallelism based on the VRAM we actually got from Colab.

In [ ]:
import subprocess

out = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits']
).decode()
name, total_mib = out.strip().split('\n')[0].split(',')
GPU_NAME = name.strip()
VRAM_GB = int(total_mib) // 1024
print(f'GPU: {GPU_NAME}  |  VRAM: {VRAM_GB} GB')

# Auto-tune config based on VRAM.
#
# Quant sizing reference (Gemma 4 31B / Qwen 3.6 35B-A3B):
#   Q4_K_M  ~18–20 GB   (baseline, fits L4 24GB)
#   Q5_K_M  ~22–24 GB
#   Q6_K    ~25–28 GB   (excellent quality)
#   Q8_0    ~33–36 GB   (near-lossless vs BF16)
#   BF16    ~62–68 GB
#
# We pick the highest-quality quant that still leaves comfortable headroom
# for activations, KV cache, and batch. For synthesis (one-shot, quality-
# critical), this matters more than for training.
if VRAM_GB >= 70:        # A100 80GB — premium allocation
    QUANT_TAG = 'q8_0'
    N_CTX = 16384
    N_BATCH = 2048
    PER_BUCKET = 500
    TIER = 'A100 80GB (Q8_0 near-lossless)'
elif VRAM_GB >= 35:       # A100 40GB
    QUANT_TAG = 'q4_k_m'
    N_CTX = 8192
    N_BATCH = 1024
    PER_BUCKET = 500
    TIER = 'A100 40GB (Q4_K_M)'
elif VRAM_GB >= 22:       # L4 24GB — tight fit, reduce context
    QUANT_TAG = 'q4_k_m'
    N_CTX = 3072
    N_BATCH = 512
    PER_BUCKET = 250
    TIER = 'L4 24GB (Q4_K_M)'
elif VRAM_GB >= 14:       # T4 16GB — won't fit 30B, warning
    QUANT_TAG = 'q4_k_s'
    N_CTX = 2048
    N_BATCH = 256
    PER_BUCKET = 100
    TIER = 'T4 16GB (WARNING: 30B model likely OOMs, switch runtime)'
else:
    raise RuntimeError(f'GPU with {VRAM_GB} GB VRAM is too small for this notebook')

# Manual override — set any of these to force a specific config.
# Useful if you want to force Q8 on a 40GB card (tight but possible) or
# downgrade to Q4 on 80GB for speed.
# QUANT_TAG = 'q6_k'
# N_CTX = 4096

print(f'tier:       {TIER}')
print(f'quant:      {QUANT_TAG}')
print(f'n_ctx:      {N_CTX}')
print(f'n_batch:    {N_BATCH}')
print(f'per_bucket: {PER_BUCKET}')

## 3. Clone the SAGE repo

Reuses prompt templates and JSONL schema from the main repo so prompt tweaks propagate automatically.

In [ ]:
%cd /content
!rm -rf sage
!git clone https://github.com/LettuceAI/sage.git
%cd sage
import sys
sys.path.insert(0, '/content/sage')

## 4. Pick a model

Set `MODEL_CHOICE` to either `qwen36` or `gemma4`. Defaults to `gemma4` — its dense architecture produces more coherent multi-turn dialogue, which matters for grooming-escalation and self-harm-buildup synthesis. Swap to `qwen36` if you hit refusals or need faster throughput.

In [ ]:
MODEL_CHOICE = 'gemma4'  # or 'qwen36'

MODEL_CONFIGS = {
    'gemma4': {
        'repo': 'llmfan46/gemma-4-31B-it-uncensored-heretic-GGUF',
        'description': 'Gemma 4 31B dense, heretic abliterated — strong dialogue quality',
    },
    'qwen36': {
        'repo': 'HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive',
        'description': 'Qwen 3.6 35B-A3B MoE, aggressive abliterated — fastest (3B active)',
    },
}

assert MODEL_CHOICE in MODEL_CONFIGS, f'unknown choice: {MODEL_CHOICE}'
MCFG = MODEL_CONFIGS[MODEL_CHOICE]
print(f'model: {MODEL_CHOICE}')
print(f'repo:  {MCFG["repo"]}')
print(f'quant: {QUANT_TAG}  (set in cell 4 based on VRAM)')
print(f'note:  {MCFG["description"]}')

## 5. Install llama-cpp-python with CUDA wheels

Prebuilt CUDA 12 wheels — no compile step. Switch to `cu118` if your Colab runtime reports CUDA 11.

In [ ]:
!pip install -q huggingface_hub
!pip install -q llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

## 6. Download the GGUF

Downloads the quant selected in cell 2. For 80GB A100 you'll be pulling ~33–36 GB of Q8_0 weights — first session ~20–30 min to download, subsequent sessions mmap from Drive cache and skip the download entirely.

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

files = list_repo_files(MCFG['repo'])
quant = QUANT_TAG.lower()
candidates = sorted(f for f in files if quant in f.lower() and f.lower().endswith('.gguf'))
print(f'{quant.upper()} candidates in {MCFG["repo"]}:')
for f in candidates:
    print(' ', f)
assert candidates, (
    f'no {quant.upper()} files found in repo — check HF for available quants '
    f'or override QUANT_TAG in cell 4'
)
GGUF_FILE = candidates[0]
print(f'\nselected: {GGUF_FILE}')

gguf_path = hf_hub_download(
    repo_id=MCFG['repo'],
    filename=GGUF_FILE,
    cache_dir=MODEL_CACHE,
)
print(f'\nmodel at: {gguf_path}')
print(f'size:     {os.path.getsize(gguf_path) / 1e9:.1f} GB')

## 7. Load the model with full GPU offload

`n_ctx` and `n_batch` were auto-selected in cell 2 based on detected VRAM. If the load OOMs on L4, lower `N_CTX` to 2048 and re-run this cell.

In [ ]:
from llama_cpp import Llama
import time

t0 = time.time()
llm = Llama(
    model_path=gguf_path,
    n_ctx=N_CTX,
    n_gpu_layers=-1,
    n_batch=N_BATCH,
    use_mmap=True,
    use_mlock=False,
    verbose=False,
    seed=42,
)
print(f'loaded in {time.time() - t0:.1f}s')

## 8. Wire into SAGE's Generator interface — with grammar-constrained JSON

Uses llama.cpp's GBNF grammar feature to **force** the model to emit valid JSON matching our schema. The sampler only permits tokens that keep the output valid at every step. No more refusals-disguised-as-structure-errors, no more `'turns'` KeyErrors, no more `"role": "therapist"`. The model literally cannot output anything that doesn't match.

Slightly slower generation (~10–15% overhead from the grammar-constrained sampler) but eliminates the entire class of parsing bugs.

In [ ]:
import json
from llama_cpp.llama_grammar import LlamaGrammar

from training.data.synthetic.generator import Generator

# JSON schema mirroring SyntheticBuilder._row_to_example's expectations.
# The grammar built from this schema constrains the sampler so the model
# literally cannot emit anything that doesn't match.
SYNTHESIS_SCHEMA = {
    "type": "array",
    "minItems": 1,
    "maxItems": 20,
    "items": {
        "type": "object",
        "properties": {
            "turns": {
                "type": "array",
                "minItems": 2,
                "maxItems": 10,
                "items": {
                    "type": "object",
                    "properties": {
                        "role": {"type": "string", "enum": ["user", "char"]},
                        "text": {"type": "string", "minLength": 1},
                    },
                    "required": ["role", "text"],
                },
            },
            "notes": {"type": "string"},
        },
        "required": ["turns", "notes"],
    },
}

SYNTHESIS_GRAMMAR = LlamaGrammar.from_json_schema(json.dumps(SYNTHESIS_SCHEMA))
print('grammar compiled from JSON schema')


class LlamaCppGenerator(Generator):
    def __init__(self, llm, grammar=None):
        self.llm = llm
        self.grammar = grammar

    def generate(
        self, system: str, user: str, *, max_tokens: int = 2048, temperature: float = 0.9
    ) -> str:
        resp = self.llm.create_chat_completion(
            messages=[
                {'role': 'system', 'content': system},
                {'role': 'user', 'content': user},
            ],
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=0.95,
            grammar=self.grammar,
        )
        return resp['choices'][0]['message']['content']


generator = LlamaCppGenerator(llm, grammar=SYNTHESIS_GRAMMAR)

# Sanity check — the output MUST be valid JSON matching our schema
sample = generator.generate(
    system='You are a helpful assistant.',
    user='Return an array containing 1 object with keys "turns" (an array of 2 objects '
         'with keys "role" (user or char) and "text") and "notes" (a string).',
    max_tokens=500,
    temperature=0.3,
)
print('sample output:')
print(sample)
parsed = json.loads(sample)  # should succeed — grammar guarantees valid JSON
assert isinstance(parsed, list), 'grammar violated: expected array'
assert 'turns' in parsed[0], 'grammar violated: missing turns'
assert parsed[0]['turns'][0]['role'] in ('user', 'char'), 'grammar violated: bad role'
print('\n✓ schema check passed — structure is guaranteed')

## 9. Synthesis with incremental save + resume

This cell is the one to re-run after any disconnect. It:
1. Checks each bucket's pending JSONL to see how many examples are already written
2. Skips buckets that are complete
3. For in-progress buckets, resumes from where the file left off
4. Appends each batch of 5 examples to disk before moving on — so the most you can lose to a disconnect is one batch
5. Reports ETA per bucket based on observed throughput

In [ ]:
import json
import time
from pathlib import Path

from training.data.synthetic.builder import SyntheticBuilder
from sage.schema import CATEGORIES

# Buckets covered by existing prompts. sexual_minors is POLICY-RESTRICTED to
# negatives only — matches SAGE's safety policy.
BUCKETS: list[tuple[str, str]] = [
    ('grooming',      'positive'),
    ('grooming',      'negative'),
    ('self_harm',     'positive'),
    ('self_harm',     'negative'),
    ('sexual_minors', 'negative'),
]

BATCH_SIZE = 5   # examples per LLM call — kept small so incremental saves survive disconnect
builder = SyntheticBuilder(generator)

def _count_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open() as f:
        return sum(1 for line in f if line.strip())

def _serialize(ex):
    return {
        'conversation': ex.conversation.to_dict(),
        'labels': {c.value: ex.labels.get(c, 0.0) for c in CATEGORIES},
        'source': ex.source,
        'meta': ex.meta,
    }

run_start = time.time()

for category, polarity in BUCKETS:
    out_path = Path(PENDING_DIR) / f'{category}_{polarity}.jsonl'
    existing = _count_lines(out_path)
    target = PER_BUCKET
    remaining = target - existing

    print(f'\n=== {category} / {polarity}  [{existing}/{target}] ===')
    if remaining <= 0:
        print('  already complete, skipping')
        continue

    bucket_start = time.time()
    n_written = existing
    errors_this_bucket: list[str] = []

    # Generate in small batches, append after each.
    while n_written < target:
        this_batch = min(BATCH_SIZE, target - n_written)
        t0 = time.time()
        batch = builder.build(
            category=category,
            polarity=polarity,
            n=this_batch,
            batch_size=this_batch,
            max_tokens=3500,
            temperature=0.95,
        )
        with out_path.open('a', encoding='utf-8') as f:
            for ex in batch.examples:
                f.write(json.dumps(_serialize(ex), ensure_ascii=False) + '\n')
        n_written += len(batch.examples)
        errors_this_bucket.extend(batch.errors)

        elapsed_bucket = time.time() - bucket_start
        done_this_session = n_written - existing
        pending = target - n_written
        rate = done_this_session / max(1, elapsed_bucket)
        eta = pending / rate if rate > 0 else 0
        print(
            f'  {n_written}/{target}'
            f'  (+{len(batch.examples)} in {time.time() - t0:.0f}s)'
            f'  rate={rate:.2f}/s  ETA={eta / 60:.1f}min'
        )

    bucket_elapsed = time.time() - bucket_start
    print(f'  ✓ done in {bucket_elapsed / 60:.1f}min  → {out_path}')
    if errors_this_bucket:
        print(f'  {len(errors_this_bucket)} errors (first 3):')
        for e in errors_this_bucket[:3]:
            print(f'    - {e[:160]}')

total = time.time() - run_start
print(f'\n[done] total session time {total / 60:.1f}min')

## 10. Preview a handful of generated examples

In [ ]:
import json
from pathlib import Path

for jsonl in sorted(Path(PENDING_DIR).glob('*.jsonl')):
    n_lines = sum(1 for _ in jsonl.open())
    print(f'\n--- {jsonl.name}  ({n_lines} examples) ---')
    with jsonl.open() as f:
        for i, line in enumerate(f):
            if i >= 2: break
            row = json.loads(line)
            turns = row['conversation']['turns']
            labels = {k: v for k, v in row['labels'].items() if v > 0}
            print(f'  Example {i + 1}  labels={labels}  turns={len(turns)}')
            for t in turns:
                text = t['text'] if len(t['text']) < 120 else t['text'][:117] + '...'
                print(f'    {t["role"]:<6}: {text}')

## 11. If disconnected — just re-run cell 9

The synthesis loop is idempotent-ish: it counts existing lines in each bucket's JSONL and picks up from there. You don't need to re-download the model (cached on Drive) or re-install anything if you're in the same session. If you're in a fresh session, run cells 1 → 9 top-to-bottom; cells 5–7 will be fast because of the Drive cache.

## 12. Next steps — on your local machine

Download the JSONL files under `/content/drive/MyDrive/sage/synthetic/pending/` to your local `data/synthetic/pending/` directory, then:

1. **Human review** — open each file and flip `meta.review_status` from `"pending"` to `"approved"` or `"rejected"` per row. Reject anything templated, off-topic, or poorly framed. Expect to keep ~70–85% of output.
2. **Merge approved rows:**
   ```bash
   python -m training.data.synthetic.cli merge \
       --in "data/synthetic/pending/*.jsonl" \
       --out data/processed/synthetic.jsonl
   ```
3. **Concatenate into training set:**
   ```bash
   cat data/processed/synthetic.jsonl >> data/processed/train.jsonl
   ```
4. **Re-trajectorize** so the new synthetic rows also go through context augmentation:
   ```bash
   python -m training.data.trajectorize_cli --in data/processed/train.jsonl --out data/processed/train.traj.jsonl
   ```
5. **Stage-2 fine-tune** from the existing v1 checkpoint at 10× lower learning rates:
   ```bash
   python -m training.train \
       --train data/processed/train.traj.jsonl \
       --val   data/processed/val.traj.jsonl \
       --out   checkpoints/sage-v1.1 \
       --resume-from checkpoints/sage-v1/best.pt \
       --epochs 1 \
       --lr-head 2e-5 --lr-top 8e-6 --lr-mid 3e-6 --lr-bot 1e-6 \
       --batch-size 4 --grad-accum 8 --max-length 256
   ```

That's v1.1.